# Multi-table synthetic data privacy comparison

This notebook applies the privacy measures from `evaluation/privacy/privacy_evaluation_pipeline.ipynb` and the black-box DCR membership attack from `evaluation/privacy/membership_inference_attack.ipynb` to each table in one or more multi-table synthetic runs. The measures are **per-table**; they do not measure relational privacy across linked records.

Use the configuration cell to switch the real seed data, select tables, and add any number of synthetic runs. Runs are compared only with seed data they were generated from. Tables missing a seed feature are reported as skipped rather than scored on a misleading partial schema. For a meaningful membership attack, set the real training and holdout directories to the actual partitions used by that synthesis run.

**Interpretation:** these are empirical risk indicators, not a differential-privacy guarantee or proof that data is private. Membership inference is meaningful only when the non-member holdout was not used to train the synthesizer. If you rely on the automatic random split below, use it only when the synthesizer was trained on the resulting training partition; a split made after synthesis does not create a valid non-member set.

## Configure the comparison

To add a run, append an entry to `SYNTHETIC_DATASETS`. `root` is relative to the repository root unless absolute, and `path_template` must resolve to one CSV per table. `REAL_TRAIN_DIR` and `REAL_HOLDOUT_DIR` can point at separate directories containing `{table}.csv` files. If no independent holdout is set, the real training directory is reproducibly split; that split is valid for membership claims only when synthesis used the corresponding training partition.

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from midst_toolkit.evaluation.privacy import (
    DistanceToClosestRecordScore,
    EpsilonIdentifiabilityRisk,
    HittingRate,
    MedianDistanceToClosestRecordScore,
    NearestNeighborDistanceRatio,
)
from midst_toolkit.evaluation.privacy.distance_preprocess import preprocess_for_distance_computation
from midst_toolkit.evaluation.privacy.distance_utils import NormType
from midst_toolkit.evaluation.privacy.epsilon_identifiability_risk import EpsilonIdentifiabilityNorm
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

def set_project_root(marker='pyproject.toml'):
    path = Path.cwd()
    for parent in (path, *path.parents):
        if (parent / marker).exists():
            os.chdir(parent)
            return parent
    raise FileNotFoundError(f'Could not find {marker} in any parent directory')


ROOT = set_project_root()
from implementations.tabular_data.evaluation.preprocessing import syntheval_preprocess

IMPLEMENTATION_ROOT = ROOT / 'implementations' / 'tabular_data'
MULTI_TABLE_ROOT = IMPLEMENTATION_ROOT / 'multi_table'
DATA_ROOT = MULTI_TABLE_ROOT / 'data'

# Choose either available real-data seed directory.
SEED_DATASET = 'berka'  # 'berka' or 'berka_feature_engineered'
REAL_TRAIN_DIR = DATA_ROOT / SEED_DATASET
REAL_HOLDOUT_DIR = None  # Example: DATA_ROOT / 'berka_holdout'
HOLDOUT_FRACTION = 0.2  # Used only when REAL_HOLDOUT_DIR is None
RANDOM_SEED = 42
TABLES = None  # None evaluates all tables with domain metadata; or e.g. ['trans', 'loan']

# Caps bound memory and quadratic distance work. Set to None to use all rows.
MAX_REAL_ROWS_PER_SPLIT = 5000
MAX_SYNTHETIC_ROWS = 10000

# Add future runs here. Each root can be absolute or relative to ROOT.
SYNTHETIC_DATASETS = [
    {
        'name': 'results/ClavaDDPM_notebook',
        'seed_dataset': 'berka',
        'root': MULTI_TABLE_ROOT / 'results',
        'path_template': 'ClavaDDPM_notebook/{table}/synthetic_final/{table}_synthetic.csv',
    },
    {
        'name': 'results2/ClavaDDPM_notebook',
        'seed_dataset': 'berka',
        'root': MULTI_TABLE_ROOT / 'results2',
        'path_template': 'ClavaDDPM_notebook/{table}/synthetic_final/{table}_synthetic.csv',
    },
    {
        'name': 'results/feature_100k_run1/ClavaDDPM_experiment',
        'seed_dataset': 'berka_feature_engineered',
        'root': MULTI_TABLE_ROOT / 'results',
        'path_template': 'feature_100k_run1/ClavaDDPM_experiment/{table}/synthetic_final/{table}_synthetic.csv',
    },
]

EXPORT_RESULTS = True
RESULTS_PATH = MULTI_TABLE_ROOT / 'evaluation' / f'privacy_results_seed-{SEED_DATASET}.csv'

if SEED_DATASET not in {'berka', 'berka_feature_engineered'}:
    raise ValueError("SEED_DATASET must be 'berka' or 'berka_feature_engineered'")
if not 0 < HOLDOUT_FRACTION < 1:
    raise ValueError('HOLDOUT_FRACTION must be between 0 and 1')
if not REAL_TRAIN_DIR.is_dir():
    raise FileNotFoundError(f'Real training data directory does not exist: {REAL_TRAIN_DIR}')
if REAL_HOLDOUT_DIR is not None and not Path(REAL_HOLDOUT_DIR).is_dir():
    raise FileNotFoundError(f'Real holdout directory does not exist: {REAL_HOLDOUT_DIR}')
if MAX_REAL_ROWS_PER_SPLIT is not None and MAX_REAL_ROWS_PER_SPLIT < 1:
    raise ValueError('MAX_REAL_ROWS_PER_SPLIT must be positive or None')
if MAX_SYNTHETIC_ROWS is not None and MAX_SYNTHETIC_ROWS < 1:
    raise ValueError('MAX_SYNTHETIC_ROWS must be positive or None')
if len({item['name'] for item in SYNTHETIC_DATASETS}) != len(SYNTHETIC_DATASETS):
    raise ValueError('Synthetic dataset names must be unique')

if REAL_HOLDOUT_DIR is None:
    print('WARNING: using a row-level split of the seed data. MIA non-members are valid only if synthesis used the corresponding training split, not the full seed data.')

/home/coder/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load schemas and prepare reusable helpers

Feature columns and their continuous/discrete types come from each table's `*_domain.json`. Identifier columns are not in those domains, so they are excluded from the distance calculations.

In [2]:
def table_names_to_evaluate():
    available = sorted(path.name.removesuffix('_domain.json') for path in REAL_TRAIN_DIR.glob('*_domain.json'))
    if TABLES is None:
        return available
    unknown = sorted(set(TABLES) - set(available))
    if unknown:
        raise ValueError(f'Tables have no domain metadata under {REAL_TRAIN_DIR}: {unknown}')
    return list(TABLES)


def load_domain(table_name):
    path = REAL_TRAIN_DIR / f'{table_name}_domain.json'
    with path.open() as file:
        domain = json.load(file)
    unsupported = {name: spec.get('type') for name, spec in domain.items() if spec.get('type') not in {'continuous', 'discrete'}}
    if unsupported:
        raise ValueError(f'Unsupported domain types for {table_name}: {unsupported}')
    return domain


def read_domain_features(path, feature_columns, context):
    if not path.is_file():
        raise FileNotFoundError(f'{context} CSV does not exist: {path}')
    available_columns = set(pd.read_csv(path, nrows=0).columns)
    missing = [column for column in feature_columns if column not in available_columns]
    if missing:
        raise ValueError(f'{context} is missing domain feature columns: {missing}')
    return pd.read_csv(path, usecols=feature_columns)[feature_columns]


def cap_rows(data, limit, seed):
    if limit is None or len(data) <= limit:
        return data.reset_index(drop=True)
    return data.sample(n=limit, random_state=seed).reset_index(drop=True)


def real_train_and_holdout(table_name, feature_columns):
    train_path = REAL_TRAIN_DIR / f'{table_name}.csv'
    real_data = read_domain_features(train_path, feature_columns, f'Real {table_name}')
    if REAL_HOLDOUT_DIR is not None:
        holdout_path = Path(REAL_HOLDOUT_DIR) / f'{table_name}.csv'
        holdout = read_domain_features(holdout_path, feature_columns, f'Holdout {table_name}')
        train = real_data
    else:
        if len(real_data) < 2:
            raise ValueError(f'Need at least two real rows to split {table_name}')
        train, holdout = train_test_split(
            real_data,
            test_size=HOLDOUT_FRACTION,
            random_state=RANDOM_SEED,
        )
    if train.empty or holdout.empty:
        raise ValueError(f'Real train and holdout must both have rows for {table_name}')
    train = cap_rows(train, MAX_REAL_ROWS_PER_SPLIT, RANDOM_SEED)
    holdout = cap_rows(holdout, MAX_REAL_ROWS_PER_SPLIT, RANDOM_SEED + 1)
    return train, holdout


def metadata_for_domain(domain, feature_columns):
    numerical = [name for name in feature_columns if domain[name]['type'] == 'continuous']
    categorical = [name for name in feature_columns if domain[name]['type'] == 'discrete']
    return {
        'numerical': numerical,
        'categorical': categorical,
        'meta_info': {
            'num_col_idx': [feature_columns.index(name) for name in numerical],
            'cat_col_idx': [feature_columns.index(name) for name in categorical],
        },
    }


def append_metric_rows(rows, dataset_name, table_name, metric_name, values):
    def add_values(prefix, value):
        if isinstance(value, dict):
            for key, nested_value in value.items():
                add_values(f'{prefix}.{key}' if prefix else str(key), nested_value)
            return
        if not np.isscalar(value):
            raise TypeError(f'Expected scalar output for {metric_name}.{prefix}, received {type(value).__name__}')
        scalar = value.item() if isinstance(value, np.generic) else value
        rows.append({
            'synthetic_dataset': dataset_name,
            'table': table_name,
            'metric': metric_name,
            'metric_key': prefix,
            'value': scalar,
        })

    add_values('', values)


def min_l2_distance(query, reference, batch_size=1024):
    if query.size(0) == 0 or reference.size(0) == 0:
        raise ValueError('Membership attack requires non-empty query and synthetic data')
    distances = []
    for start in range(0, query.size(0), batch_size):
        batch = query[start:start + batch_size]
        distances.append(torch.cdist(batch, reference, p=2).min(dim=1).values.cpu().numpy())
    return np.concatenate(distances)


def run_dcr_membership_attack(distance_train, distance_synthetic, distance_holdout):
    rng = np.random.default_rng(RANDOM_SEED)
    n_each = min(len(distance_train), len(distance_holdout))
    if n_each < 2:
        raise ValueError(f'Membership attack needs at least two member and two non-member rows; got {n_each} each')
    member_idx = rng.choice(len(distance_train), size=n_each, replace=False)
    non_member_idx = rng.choice(len(distance_holdout), size=n_each, replace=False)
    candidates = pd.concat([
        distance_train.iloc[member_idx].reset_index(drop=True),
        distance_holdout.iloc[non_member_idx].reset_index(drop=True),
    ], ignore_index=True)
    labels = np.concatenate([np.ones(n_each, dtype=int), np.zeros(n_each, dtype=int)])
    candidate_tensor = torch.tensor(candidates.to_numpy(), dtype=torch.float32)
    synthetic_tensor = torch.tensor(distance_synthetic.to_numpy(), dtype=torch.float32)
    dcr = min_l2_distance(candidate_tensor, synthetic_tensor)
    scores = -dcr  # Smaller DCR is stronger evidence that a row was a member.
    auc = roc_auc_score(labels, scores)
    fpr, tpr, _ = roc_curve(labels, scores)
    eligible = np.flatnonzero(fpr <= 0.10)
    tpr_at_10_fpr = float(tpr[eligible[-1]]) if len(eligible) else 0.0
    calibration_idx, test_idx = train_test_split(
        np.arange(len(labels)),
        test_size=0.5,
        random_state=RANDOM_SEED,
        stratify=labels,
    )
    calibration_fpr, calibration_tpr, thresholds = roc_curve(labels[calibration_idx], scores[calibration_idx])
    threshold = thresholds[np.argmax(calibration_tpr - calibration_fpr)]
    predictions = (scores[test_idx] >= threshold).astype(int)
    return {
        'roc_auc': float(auc),
        'tpr_at_10_fpr': tpr_at_10_fpr,
        'calibrated_test_accuracy': float(accuracy_score(labels[test_idx], predictions)),
        'mean_member_dcr': float(dcr[labels == 1].mean()),
        'mean_non_member_dcr': float(dcr[labels == 0].mean()),
        'candidates_per_class': int(n_each),
    }


table_names = table_names_to_evaluate()
if not table_names:
    raise ValueError(f'No tables with domain metadata found under {REAL_TRAIN_DIR}')
selected_datasets = [item for item in SYNTHETIC_DATASETS if item['seed_dataset'] == SEED_DATASET]
if not selected_datasets:
    raise ValueError(f'No synthetic run is configured for seed dataset {SEED_DATASET!r}')
print(f'Evaluating {len(table_names)} tables against {len(selected_datasets)} run(s) using {SEED_DATASET!r}.')

Evaluating 8 tables against 2 run(s) using 'berka'.


## Calculate per-table privacy measures

Hitting Rate, Epsilon Identifiability Risk, DCR, median DCR, and NNDR use the same implementations as the single-table privacy pipeline. The membership attack is black-box and uses only distances to the synthetic records. Compare values **within the same table and schema**; metric scales are not directly comparable between different tables.

In [3]:
metric_rows = []
skipped = []

for table_name in table_names:
    domain = load_domain(table_name)
    feature_columns = list(domain)
    feature_types = metadata_for_domain(domain, feature_columns)
    real_train, real_holdout = real_train_and_holdout(table_name, feature_columns)
    numerical_columns = feature_types['numerical']
    categorical_columns = feature_types['categorical']
    meta_info = feature_types['meta_info']

    for dataset_config in selected_datasets:
        dataset_name = dataset_config['name']
        root = Path(dataset_config['root'])
        if not root.is_absolute():
            root = ROOT / root
        synthetic_path = root / dataset_config['path_template'].format(table=table_name)
        if not synthetic_path.is_file():
            skipped.append({
                'synthetic_dataset': dataset_name,
                'table': table_name,
                'reason': f'CSV not found: {synthetic_path}',
            })
            continue

        synthetic_header = set(pd.read_csv(synthetic_path, nrows=0).columns)
        missing_features = [name for name in feature_columns if name not in synthetic_header]
        if missing_features:
            skipped.append({
                'synthetic_dataset': dataset_name,
                'table': table_name,
                'reason': f'missing seed-domain features: {missing_features}',
            })
            continue
        synthetic_data = read_domain_features(synthetic_path, feature_columns, f'Synthetic {dataset_name}/{table_name}')
        if synthetic_data.empty:
            skipped.append({
                'synthetic_dataset': dataset_name,
                'table': table_name,
                'reason': 'synthetic CSV has no rows',
            })
            continue
        synthetic_data = cap_rows(synthetic_data, MAX_SYNTHETIC_ROWS, RANDOM_SEED)

        syntheval_train, syntheval_synthetic, syntheval_holdout = syntheval_preprocess(
            numerical_columns,
            categorical_columns,
            real_train,
            synthetic_data,
            real_holdout,
        )

        hitting_rate = HittingRate(
            categorical_columns=categorical_columns,
            numerical_columns=numerical_columns,
            hitting_threshold=0.03,
            do_preprocess=False,
        ).compute(syntheval_train, syntheval_synthetic)
        append_metric_rows(metric_rows, dataset_name, table_name, 'hitting_rate', hitting_rate)

        epsilon_risk = EpsilonIdentifiabilityRisk(
            categorical_columns=categorical_columns,
            numerical_columns=numerical_columns,
            norm=EpsilonIdentifiabilityNorm('gower'),
            do_preprocess=False,
        ).compute(syntheval_train, syntheval_synthetic)
        append_metric_rows(metric_rows, dataset_name, table_name, 'epsilon_identifiability_risk', epsilon_risk)

        distance_train, distance_synthetic, distance_holdout = preprocess_for_distance_computation(
            meta_info=meta_info,
            real_data_train=real_train,
            synthetic_data=synthetic_data,
            real_data_test=real_holdout,
        )
        batch_size = 1000
        dcr = DistanceToClosestRecordScore(NormType('l2'), batch_size, do_preprocess=False).compute(
            distance_train,
            distance_synthetic,
            distance_holdout,
        )
        append_metric_rows(metric_rows, dataset_name, table_name, 'dcr', dcr)

        try:
            median_dcr = MedianDistanceToClosestRecordScore(
                NormType("l2"),
                batch_size,
                do_preprocess=False,
            ).compute(distance_train, distance_synthetic)
        except ZeroDivisionError:
            skipped.append({
                "synthetic_dataset": dataset_name,
                "table": table_name,
                "reason": (
                    "Median DCR is undefined: the median real-to-real "
                    "nearest-neighbor distance is zero."
                ),
            })
        else:
            append_metric_rows(
                metric_rows, dataset_name, table_name, "median_dcr", median_dcr
            )

        nndr = NearestNeighborDistanceRatio(NormType('l2'), batch_size, do_preprocess=False).compute(
            distance_train,
            distance_synthetic,
            distance_holdout,
        )
        append_metric_rows(metric_rows, dataset_name, table_name, 'nndr', nndr)

        if min(len(distance_train), len(distance_holdout)) < 2:
            skipped.append({
                'synthetic_dataset': dataset_name,
                'table': table_name,
                'reason': 'Membership inference attack needs at least two member and two non-member rows after preprocessing',
            })
        else:
            mia = run_dcr_membership_attack(distance_train, distance_synthetic, distance_holdout)
            append_metric_rows(metric_rows, dataset_name, table_name, 'dcr_membership_inference', mia)

        print(f'Completed {dataset_name} / {table_name}')

privacy_results = pd.DataFrame(metric_rows)
if privacy_results.empty:
    print('No privacy metrics were computed. Check the selected seed dataset and synthetic paths.')
else:
    display(privacy_results.sort_values(['table', 'metric', 'synthetic_dataset', 'metric_key']))

if skipped:
    print('Skipped tables or metrics:')
    display(pd.DataFrame(skipped))

Computing nearest neighbor distances from real/holdout dataset to synthetic dataset.:   0%|          | 0/31 [00:00<?, ?it/s]/home/coder/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/syntheval/utils/nn_distance.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  b[num_cols] = b[num_cols].astype("float")
/home/coder/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/syntheval/utils/nn_distance.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  b[num_cols] = b[num_cols].astype("float"

Completed results/ClavaDDPM_notebook / account


Computing nearest neighbor distances from real/holdout dataset to synthetic dataset.:   0%|          | 0/36 [00:00<?, ?it/s]/home/coder/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/syntheval/utils/nn_distance.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  b[num_cols] = b[num_cols].astype("float")
/home/coder/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/syntheval/utils/nn_distance.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  b[num_cols] = b[num_cols].astype("float"

Completed results2/ClavaDDPM_notebook / account


Computing nearest neighbor distances from real/holdout dataset to synthetic dataset.:   0%|          | 0/6 [00:00<?, ?it/s]/home/coder/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/syntheval/utils/nn_distance.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  b[num_cols] = b[num_cols].astype("float")
/home/coder/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/syntheval/utils/nn_distance.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  b[num_cols] = b[num_cols].astype("float")

...

Computing nearest neighbor distances from real/holdout dataset to synthetic dataset.:   0%|          | 0/79 [00:00<?, ?it/s]/home/coder/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/syntheval/utils/nn_distance.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  b[num_cols] = b[num_cols].astype("float")
/home/coder/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/syntheval/utils/nn_distance.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  b[num_cols] = b[num_cols].astype("float"

Completed results2/ClavaDDPM_notebook / trans


,synthetic_dataset,table,metric,metric_key,value
2,results/ClavaDDPM_notebook,account,dcr,dcr_score,0.526818
15,results2/ClavaDDPM_notebook,account,dcr,dcr_score,0.527594
9,results/ClavaDDPM_notebook,account,dcr_membership_inference,calibrated_test_accuracy,0.506667
12,results/ClavaDDPM_notebook,account,dcr_membership_inference,candidates_per_class,900.000000
10,results/ClavaDDPM_notebook,account,dcr_membership_inference,mean_member_dcr,1.414213
...,...,...,...,...,...
199,results/ClavaDDPM_notebook,trans,nndr,privacy_loss_standard_error,0.000003
210,results2/ClavaDDPM_notebook,trans,nndr,mean_nndr,0.999942
211,results2/ClavaDDPM_notebook,trans,nndr,nndr_standard_error,0.000002
212,results2/ClavaDDPM_notebook,trans,nndr,privacy_loss,0.000007


Skipped tables or metrics:


,synthetic_dataset,table,reason
0,results/ClavaDDPM_notebook,account,Median DCR is undefined: the median real-to-re...
1,results2/ClavaDDPM_notebook,account,Median DCR is undefined: the median real-to-re...
2,results/ClavaDDPM_notebook,disp,Median DCR is undefined: the median real-to-re...
3,results2/ClavaDDPM_notebook,disp,Median DCR is undefined: the median real-to-re...


## Compare runs and optionally export

The pivot is convenient for comparing runs on the same table/metric. Lower Hitting Rate and Epsilon Identifiability Risk generally indicate less observed disclosure. For membership inference, ROC-AUC closer to 0.5 and TPR/accuracy closer to chance indicate a weaker attack. Interpret DCR and NNDR keys using the source privacy notebook. No single score establishes privacy.

In [4]:
if not privacy_results.empty:
    comparison = privacy_results.pivot_table(
        index=['table', 'metric', 'metric_key'],
        columns='synthetic_dataset',
        values='value',
        aggfunc='first',
    )
    display(comparison)

if EXPORT_RESULTS:
    RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    privacy_results.to_csv(RESULTS_PATH, index=False)
    print(f'Wrote privacy metrics to {RESULTS_PATH}')

synthetic_dataset                                             results/ClavaDDPM_notebook  \
table   metric                   metric_key                                                
account dcr                      dcr_score                                      0.526818   
        dcr_membership_inference calibrated_test_accuracy                       0.506667   
                                 candidates_per_class                         900.000000   
                                 mean_member_dcr                                1.414213   
                                 mean_non_member_dcr                            1.414213   
...                                                                                  ...   
trans   median_dcr               median_dcr_score                             112.063141   
        nndr                     mean_nndr                                      0.999940   
                                 nndr_standard_error                            0.000002   
                                 privacy_loss                                   0.000006   
                                 privacy_loss_standard_error                    0.000003   

synthetic_dataset                                             results2/ClavaDDPM_notebook  
table   metric                   metric_key                                                
account dcr                      dcr_score                                       0.527594  
        dcr_membership_inference calibrated_test_accuracy                        0.506667  
                                 candidates_per_class                          900.000000  
                                 mean_member_dcr                                 1.414213  
                                 mean_non_member_dcr                             1.414213  
...                                                                                   ...  
trans   median_dcr               median_dcr_score                              112.063495  
        nndr                     mean_nndr                                       0.999942  
                                 nndr_standard_error                             0.000002  
                                 privacy_loss                                    0.000007  
                                 privacy_loss_standard_error                     0.000003  

[110 rows x 2 columns]

Wrote privacy metrics to /home/coder/synthetic-data-bootcamp/implementations/tabular_data/multi_table/evaluation/privacy_results_seed-berka.csv
